# Climate Lexicon Expansion and Validation Framework

This notebook documents the systematic construction and expansion of a domain-specific climate-finance lexicon used to identify climate-related discourse within central bank speeches. The objective is to move beyond a narrow seed-term approach and develop a comprehensive, context-relevant vocabulary suitable for large-scale text mining and econometric analysis.

The lexicon development proceeds in structured stages:

1. **Seed-Term Initialization**
   - A curated set of foundational climate and climate-finance terms is defined based on established literature and institutional terminology.
   - These initial seed terms are stored in the supplementary file `seed_terms.csv`.
   - The seed list captures core concepts related to physical risks, transition risks, mitigation, adaptation, sustainable finance, and climate-related policy frameworks.

2. **Semantic Expansion via Word Embeddings**
   - Word2Vec-based similarity methods are applied to identify semantically proximate candidate terms.
   - Candidate expansions are generated using similarity thresholds designed to preserve contextual meaning rather than relying solely on keyword frequency.
   - This stage increases recall while maintaining domain relevance.

3. **Manual Expert Review**
   - All expanded terms are manually reviewed to eliminate spurious or contextually ambiguous inclusions.
   - Financially meaningful and institutionally relevant terminology is retained.
   - Additional terminology is incorporated from prior climate-finance research to ensure theoretical alignment with the literature.
   - The final validated lexicon comprises **889 terms**.

4. **Final Lexicon Output**
   - The complete validated lexicon is provided in the supplementary material as `climate_lexicon.csv`.
   - The seed-term file used for initialization is provided as `seed_terms.csv`.
   - The lexicon is formatted for token-safe sentence matching in the downstream extraction pipeline.

The resulting lexicon forms the foundation for high-recall sentence extraction, which is subsequently refined through transformer-based classification to achieve high precision. This hybrid methodology ensures both breadth and contextual accuracy in measuring climate discourse intensity across central banks.

This notebook is designed to provide full transparency and replicability of the lexicon construction process for academic and policy research applications.

0) Install and imports

In [1]:
# !pip install -U pandas numpy gensim nltk scikit-learn

import os
import re
import string
import pandas as pd
import numpy as np

from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser

import nltk
nltk.download("punkt", quiet=True)
from nltk.tokenize import word_tokenize


1) File paths

In [5]:
SEED_PATH = "seed_terms.csv"

CANDIDATE_PATH = "candidate_sentences.csv"

OUTPUT_DIR = "lexicon_expansion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


2) Load data

In [6]:
seed_df = pd.read_csv(SEED_PATH)
cand_df = pd.read_csv(CANDIDATE_PATH)

# Seed column detection
seed_col = seed_df.columns[0]
seed_terms = (
    seed_df[seed_col]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

# Sentence column detection (prefer 'sentence')
if "sentence" in cand_df.columns:
    sent_col = "sentence"
else:
    # fallback: first non-url text-like column
    non_url_cols = [c for c in cand_df.columns if c.lower() not in ("url", "link", "source")]
    sent_col = non_url_cols[0] if non_url_cols else cand_df.columns[0]

sentences_raw = (
    cand_df[sent_col]
    .dropna()
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .tolist()
)

print("Seeds:", len(seed_terms))
print("Sentences:", len(sentences_raw))
print("Sentence column used:", sent_col)


Seeds: 45
Sentences: 5697
Sentence column used: sentence


3) Text cleaning + tokenization

In [7]:
# Domain-friendly cleaning:
# - lowercase
# - keep letters, numbers, and a few separators
# - normalize hyphens to spaces
# - drop very short tokens

_keep = set(string.ascii_lowercase + string.digits + " -_/")
_ws_re = re.compile(r"\s+")

def clean_text(s: str) -> str:
    s = s.lower()
    s = s.replace("–", "-").replace("—", "-")
    s = "".join(ch if ch in _keep else " " for ch in s)
    s = s.replace("-", " ")
    s = _ws_re.sub(" ", s).strip()
    return s

def tokenize(s: str) -> list[str]:
    toks = word_tokenize(s)
    toks = [t for t in toks if len(t) >= 2]
    return toks

sentences_clean = [clean_text(s) for s in sentences_raw]
tokens_list = [tokenize(s) for s in sentences_clean]
tokens_list = [toks for toks in tokens_list if len(toks) >= 3]

print("Tokenized sentences:", len(tokens_list))


Tokenized sentences: 5697


4) Phrase detection (bigrams/trigrams)

In [10]:
from gensim.models.phrases import Phrases, Phraser
import os

# Phrase mining: detect multiword expressions, then use those phrase tokens inside Word2Vec.

bigram = Phrases(tokens_list, min_count=5, threshold=10.0, delimiter="_")
bigram_phraser = Phraser(bigram)

tokens_bigram = [bigram_phraser[toks] for toks in tokens_list]

trigram = Phrases(tokens_bigram, min_count=3, threshold=8.0, delimiter="_")
trigram_phraser = Phraser(trigram)

tokens_phrased = [trigram_phraser[toks] for toks in tokens_bigram]

# Optional save
phrase_dir = os.path.join(OUTPUT_DIR, "phrase_model")
os.makedirs(phrase_dir, exist_ok=True)
bigram_phraser.save(os.path.join(phrase_dir, "bigram.phraser"))
trigram_phraser.save(os.path.join(phrase_dir, "trigram.phraser"))

print("Example phrased tokens:", tokens_phrased[0][:30])


Example phrased tokens: ['transition', 'versus', 'physical', 'climate_risk', 'pricing', 'in', 'european', 'financial_markets', 'text', 'based', 'approach', '35']


5) Train Word2Vec

In [11]:
w2v = Word2Vec(
    sentences=tokens_phrased,
    vector_size=300,
    window=6,
    min_count=5,
    workers=os.cpu_count() or 4,
    sg=1,          # skip-gram
    negative=10,
    sample=1e-4,
    epochs=20
)

w2v_dir = os.path.join(OUTPUT_DIR, "word2vec_model")
os.makedirs(w2v_dir, exist_ok=True)
w2v.save(os.path.join(w2v_dir, "climatefincb_word2vec.model"))

print("Vocab size:", len(w2v.wv))


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocab size: 4279


6) Normalize seeds into the same “phrase space”

In [12]:
def seed_to_model_token(seed: str) -> str:
    s = clean_text(seed)
    toks = tokenize(s)
    if not toks:
        return ""
    # Apply the same phrasers to convert "climate risk" -> "climate_risk" when learned
    toks = bigram_phraser[toks]
    toks = trigram_phraser[toks]
    # If multiple tokens remain, join with "_" as a candidate phrase token
    # (but only keep if the joined form exists in the vocab)
    if len(toks) == 1:
        return toks[0]
    joined = "_".join(toks)
    return joined

seed_map = [(seed, seed_to_model_token(seed)) for seed in seed_terms]
seed_map_df = pd.DataFrame(seed_map, columns=["seed_term", "model_token"])
seed_map_df["in_vocab"] = seed_map_df["model_token"].apply(lambda t: t in w2v.wv if t else False)

print(seed_map_df["in_vocab"].value_counts())
seed_map_df.head(10)


in_vocab
True     24
False    21
Name: count, dtype: int64


,seed_term,model_token,in_vocab
0,climate finance,climate_finance,True
1,climate change,climate_change,True
2,climate mitigation finance,climate_mitigation_finance,False
3,climate adaptation finance,climate_adaptation_finance,False
4,Financial Mechanism of the United Nations Fram...,financial_mechanism_of_the_united_nations_fram...,False
5,Green Climate Fund,green_climate_fund,False
6,Special Climate Change Fund,special_climate_change_fund,False
7,Climate Investment Funds,climate_investment_funds,False
8,Strategic Climate Fund,strategic_climate_fund,False
9,green bond,green_bond,True


7) Expand lexicon using nearest neighbors

In [13]:
def token_frequency(tokens_phrased: list[list[str]]) -> pd.Series:
    flat = [t for sent in tokens_phrased for t in sent]
    return pd.Series(flat).value_counts()

freq = token_frequency(tokens_phrased)

def is_good_candidate(t: str) -> bool:
    # Filters: remove purely numeric, very short, or tokens with many digits
    if not t or len(t) < 3:
        return False
    if t.isdigit():
        return False
    digit_share = sum(ch.isdigit() for ch in t) / len(t)
    if digit_share > 0.4:
        return False
    return True

TOPN_PER_SEED = 50
MIN_SIM = 0.45

rows = []
for seed_term, model_token in seed_map:
    if not model_token or model_token not in w2v.wv:
        continue
    for cand, sim in w2v.wv.most_similar(model_token, topn=TOPN_PER_SEED):
        if sim < MIN_SIM:
            continue
        if not is_good_candidate(cand):
            continue
        rows.append({
            "seed_term": seed_term,
            "seed_token": model_token,
            "candidate_term": cand.replace("_", " "),
            "candidate_token": cand,
            "similarity": float(sim),
            "frequency": int(freq.get(cand, 0)),
            "method": "word2vec_nn"
        })

candidates_df = pd.DataFrame(rows)

# De-duplicate within a seed by taking best similarity
candidates_df = (
    candidates_df.sort_values(["seed_term", "similarity"], ascending=[True, False])
    .drop_duplicates(subset=["seed_term", "candidate_token"], keep="first")
)

# Optional: global de-duplication across seeds (keep max similarity)
candidates_global = (
    candidates_df.sort_values("similarity", ascending=False)
    .drop_duplicates(subset=["candidate_token"], keep="first")
)

print("Candidate rows (seed-level):", len(candidates_df))
print("Candidate rows (global unique):", len(candidates_global))
candidates_global.head(15)


Candidate rows (seed-level): 1187
Candidate rows (global unique): 802


,seed_term,seed_token,candidate_term,candidate_token,similarity,frequency,method
296,emissions trading,emissions_trading,carbon taxes,carbon_taxes,0.980147,19,word2vec_nn
446,financed emissions,financed_emissions,reaching net zero,reaching_net_zero,0.968769,6,word2vec_nn
297,emissions trading,emissions_trading,taxes,taxes,0.964050,14,word2vec_nn
298,emissions trading,emissions_trading,cap,cap,0.960049,10,word2vec_nn
1137,pollution,pollution,prevention,prevention,0.958375,5,word2vec_nn
447,financed emissions,financed_emissions,orderly transition,orderly_transition,0.955073,6,word2vec_nn
299,emissions trading,emissions_trading,subsidies,subsidies,0.952792,20,word2vec_nn
300,emissions trading,emissions_trading,electricity,electricity,0.951055,8,word2vec_nn
301,emissions trading,emissions_trading,instrument,instrument,0.950831,13,word2vec_nn
302,emissions trading,emissions_trading,equal,equal,0.946387,7,word2vec_nn


8) Export for manual review

In [14]:
out_path = os.path.join(OUTPUT_DIR, "expanded_lexicon_candidates.csv")
candidates_global.sort_values(["similarity", "frequency"], ascending=[False, False]).to_csv(out_path, index=False)
print("Saved:", out_path)


Saved: lexicon_expansion_outputs/expanded_lexicon_candidates.csv
